# 🎓 TA 参照用：精度向上パート 実装例集

> **このノートブックは受講生には配布しません。**
> `HousePrice_practice_student_v2.ipynb` の **「精度向上パート」**に対応する実装例集です。
> 各 FREE セルに対する実装例を豊富に掲載しています。受講生の質問や詰まりに応じてヒントを出す際の参考にしてください。

---

### 学生版 v2 からの変更点（TA 向け注意）

学生版 v2 では、精度向上パートの欠損値補完が **2 段構成** になりました。

| 変更箇所 | 内容 |
|:---|:---|
| **(A-1)** | `NUM_FILL_DEFAULT` + `FILL_CONSTANT` + `COL_FILL_MAP`（カラム別個別指定）に変更 |
| **(A-2)** | **新規追加**。`GROUP_FILL` でカテゴリ別代表値補完（セクション 4 の groupby 補完）をパイプラインに組み込めるようになった |
| (B) | 特徴量追加。`add_features()` の中身は学生版と同じ構造 |
| (C) | 特徴量選択。`THRESHOLD` と `DROP_COLS` は変更なし |

このノートブックは上記の変更を反映した **A-1 / A-2 / B / C / パイプライン（D）/ 提出ファイル（E）** の構成で実装例を示します。

---

### このノートブックの構成

| セクション | 内容 |
|:---|:---|
| 0 | セットアップ（受講生ノートブックと同一） |
| A-1 | 欠損値補完：デフォルト ＋ カラム別個別指定の実装例 |
| A-2 | 欠損値補完：カテゴリ別代表値補完（GROUP_FILL）の実装例 |
| B | 特徴量作成の実装例 |
| C | 特徴量選択の実装例 |
| D | パイプライン実行（自動評価） |
| E | 提出ファイル作成 |


---
## 0. セットアップ（受講生ノートブックと同一）


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

!pip install japanize-matplotlib -q
import japanize_matplotlib
print('✅ ライブラリ読み込み完了')


In [ ]:
PATH = 'https://raw.githubusercontent.com/Toyoda05/kaggle-house-price-practice/main/'
train = pd.read_csv(PATH + 'train.csv')
test  = pd.read_csv(PATH + 'test.csv')
test_ids = test['Id']
print(f'train: {train.shape}  /  test: {test.shape}')
train.head()


In [ ]:
# ベースライン（受講生ノートブックと同一コード）
y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice', 'Id'])
X_test = test.drop(columns=['Id']).copy()

all_data = pd.concat([X, X_test], axis=0).reset_index(drop=True)
for col in all_data.select_dtypes(include='number').columns:
    all_data[col] = all_data[col].fillna(all_data[col].median())
for col in all_data.select_dtypes(include='object').columns:
    all_data[col] = all_data[col].fillna('Missing')
all_data = pd.get_dummies(all_data, drop_first=True)

X_proc      = all_data.iloc[:len(X)].copy()
X_test_proc = all_data.iloc[len(X):].copy()

X_tr, X_vl, y_tr, y_vl = train_test_split(X_proc, y, test_size=0.2, random_state=42)
model_bl = LinearRegression()
model_bl.fit(X_tr, y_tr)
rmse_baseline = np.sqrt(mean_squared_error(y_vl, model_bl.predict(X_vl)))
print(f'🔵 ベースライン RMSE: {rmse_baseline:.5f}')


---
## A-1. 数値列の欠損補完：デフォルト ＋ カラム別個別指定

受講生が操作できる変数は `NUM_FILL_DEFAULT`・`FILL_CONSTANT`・`COL_FILL_MAP` の 3 つです。
以下に代表的なパターンと想定される効果を示します。

### TA向け解説

HousePriceデータの数値欠損列のうち、代表的なものの補完方針：

| 列 | 欠損の意味 | 推奨補完 | 理由 |
|:---|:---|:---|:---|
| `LotFrontage` | 道路幅（真の欠損） | median | 右裾分布なので中央値が安定 |
| `MasVnrArea` | 外壁面積（設備なしの可能性） | `constant_0` | 0は「外壁なし」を意味することが多い |
| `GarageYrBlt` | ガレージ建築年（ガレージなし物件） | median | 0にすると外れ値になりうる点に注意 |
| `BsmtFinSF1/2` | 地下仕上げ面積（地下なし） | `constant_0` | 地下なし = 0 が自然 |

`COL_FILL_MAP` に書いた列は **デフォルト補完より優先** される点を強調してください。

### パターン例


In [ ]:
# ── A-1-①：デフォルト方法を median にする（最も安全な設定）───────────────
# 右裾分布の多い不動産データでは median が mean より外れ値の影響を受けにくい

NUM_FILL_DEFAULT = 'median'
FILL_CONSTANT    = 0
COL_FILL_MAP     = {}   # 個別指定なし＝全列に median を適用

# 比較用：LotFrontage で平均 vs 中央値
col = 'LotFrontage'
orig = train[col].copy()
fill_mean   = orig.fillna(orig.mean())
fill_median = orig.fillna(orig.median())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, filled, label, color in zip(
        axes,
        [orig.dropna(), fill_mean, fill_median],
        ['元データ（欠損除く）', f'平均値補完 ({orig.mean():.1f})', f'中央値補完 ({orig.median():.1f})'],
        ['salmon', 'steelblue', 'green']):
    ax.hist(filled, bins=40, color=color, alpha=0.7)
    ax.set_title(label)
    ax.set_xlabel(col)
    ax.set_ylabel('度数')
plt.suptitle('LotFrontage：補完方法の比較', fontsize=13)
plt.tight_layout()
plt.show()

print(f'分散比較  元データ: {orig.var():.1f}')
print(f'         平均補完: {fill_mean.var():.1f}  （分散が縮小→データの広がりが失われる）')
print(f'         中央補完: {fill_median.var():.1f}')


In [ ]:
# ── A-1-②：COL_FILL_MAP でカラム別に個別指定する ───────────────────────
# 「外壁なし＝0」など、列ごとに方針を変える例

COL_FILL_MAP_EXAMPLE = {
    'MasVnrArea' : 'constant_0',   # 外壁仕上げなし → 0
    'LotFrontage': 'median',       # 右裾分布 → 中央値（デフォルトと同じだが明示する例）
    'GarageYrBlt': 'median',       # ガレージ建築年
}

col2 = 'OverallCond'
print('OverallCond の歪度:', train[col2].skew().round(3))
print('→ 歪度が±0.5以内なら mean も選択肢に入る（COL_FILL_MAP に "mean" を指定）')

col3 = 'MasVnrArea'
n_miss = train[col3].isnull().sum()
zero_count = (train[col3] == 0).sum()
print(f'\nMasVnrArea  欠損数: {n_miss} / ゼロ件数: {zero_count}')
print('→ 欠損 ≒ 0 と考えられる場合は COL_FILL_MAP に "constant_0" を指定するのが適切')
print()
print('【TAメモ】constant_<数値> の書式は文字列の split で数値を取り出している点を')
print('         パイプラインのコードと合わせて説明すると理解が進みます。')


---
## A-2. カテゴリ別代表値補完（GROUP_FILL）── 新規セクション

学生版 v2 で新しく追加された欄です。受講生が操作できる変数は `GROUP_FILL` のみで、
セクション 4（前処理発展）で学んだ groupby 補完をリストに追加するだけで
パイプラインに反映されます。

### TA向け解説

`GROUP_FILL` はタプルのリストで、各タプルは `(補完したい列, グループ化する列, 集計関数)` です。
パイプライン側では以下の順序で処理されます。

1. `get_dummies` 適用前に実行（グループ列が文字列のまま必要なため）
2. グループ統計量は **学習データ部分のみ**（`_train_part = all_c.iloc[:len(X_c)]`）から計算 → リーク防止
3. `GROUP_FILL` で補完できなかった残余欠損は A-1 の設定で後処理される

### 効果が出やすい組み合わせの例


In [ ]:
# ── A-2-①：Neighborhood 別中央値で LotFrontage を補完 ─────────────────
# 地区によって道路幅の傾向が異なるため、全体中央値より精緻な補完が可能

col4 = 'LotFrontage'
fill_global = train[col4].fillna(train[col4].median())
fill_group  = train[col4].fillna(train.groupby('Neighborhood')[col4].transform('median'))

print('全体中央値補完の分散 :', round(fill_global.var(), 2))
print('Neighborhood別補完の分散:', round(fill_group.var(), 2))
print('→ グループ別補完のほうが分散が大きい（元のばらつきを保てている）')
print()
print('Neighborhood ごとの LotFrontage 中央値（上位5地区）:')
print(train.groupby('Neighborhood')[col4].median().sort_values(ascending=False).head())

GROUP_FILL_EXAMPLE_1 = [
    ('LotFrontage', 'Neighborhood', 'median'),
]


In [ ]:
# ── A-2-②：GarageType 別中央値で GarageYrBlt を補完 ───────────────────
# ガレージがある物件は GarageType によって建築年の傾向が異なる

col5 = 'GarageYrBlt'
group_med = train.groupby('GarageType')[col5].median()
print('GarageType 別 GarageYrBlt 中央値:')
print(group_med)

fill_global5 = train[col5].fillna(train[col5].median())
fill_group5  = train[col5].fillna(train.groupby('GarageType')[col5].transform('median'))
print()
print('全体中央値補完の分散  :', round(fill_global5.var(), 2))
print('GarageType別補完の分散:', round(fill_group5.var(), 2))

GROUP_FILL_EXAMPLE_2 = [
    ('GarageYrBlt', 'GarageType', 'median'),
]


In [ ]:
# ── A-2-③：複数列を同時に GROUP_FILL する完全な設定例 ─────────────────
GROUP_FILL_FULL_EXAMPLE = [
    ('LotFrontage', 'Neighborhood', 'median'),  # 地域別中央値で道路幅を補完
    ('GarageYrBlt', 'GarageType',   'median'),  # ガレージ種別別でガレージ建設年を補完
    ('MasVnrArea',  'MasVnrType',   'median'),  # 外壁材種別で外壁面積を補完
    ('LotFrontage', 'LotConfig',    'median'),  # 区画形状別で道路幅を補完（※同じ列を複数回指定した場合は後の設定で上書きされる）
]

for tc, gc, af in GROUP_FILL_FULL_EXAMPLE:
    n_groups = train[gc].nunique()
    print(f'{tc:<14} ← {gc:<14}（{af}）　グループ数: {n_groups}')

print()
print('【TAメモ】')
print('・GROUP_FILL は「同じ列を複数回指定」した場合、リストの後ろの設定で上書きされます。')
print('  受講生が誤って同じ列を2回書いてしまった場合のヒントに使えます。')
print('・グループ数が極端に少ない/多いカテゴリ列（例: 1地区だけ1件しかない）では')
print('  そのグループの中央値が NaN になり、結局 A-1 のデフォルト補完に回ることも説明すると良い。')


---
## B. 特徴量作成の実装例

`add_features()` に追加する行の例を多数掲載します。
SalePrice との相関係数 r も合わせて表示しているので、
**「どの特徴量が効きそうか」を受講生に問いかける際の根拠**に使ってください。

### カテゴリ別実装例

| カテゴリ | 特徴量名 | 計算式 | 相関の期待値 |
|:---|:---|:---|:---|
| 面積系 | TotalSF | 1st+2nd+Bsmt | 高 (r≈0.8) |
| 面積系 | LivLotRatio | GrLivArea/LotArea | 中 |
| バスルーム | TotalBath | Full+0.5Half+BsmtFull+0.5BsmtHalf | 高 (r≈0.65) |
| 築年数 | RemodAge | YrSold-YearRemodAdd | 中（HouseAgeは学生版で既に作成済み） |
| 品質×面積 | QualxArea | OverallQual*GrLivArea | 最高 (r≈0.87) |
| フラグ | HasGarage | GarageCars>0 | 中 |
| フラグ | HasBsmt | TotalBsmtSF>0 | 中 |
| 総合指標 | TotalPorch | OpenPorch+EnclosedPorch+3SsnPorch+ScreenPorch | 低〜中 |

> **TAメモ**：学生版 v2 の `add_features()` には、すでに `HouseAge = YrSold - YearBuilt` が
> ベース特徴量として実装済みです。受講生が追加するのはその下の部分です。


In [ ]:
# ── B-1：面積系特徴量 ────────────────────────────────────────────────────
train_b = train.copy()

# 総床面積（最も効く組み合わせの一つ）
train_b['TotalSF'] = train_b['1stFlrSF'] + train_b['2ndFlrSF'] + train_b['TotalBsmtSF']

# 地上居住面積 ÷ 土地面積（敷地利用率）
# 0除算を避けるため clip で最小値を設定
train_b['LivLotRatio'] = train_b['GrLivArea'] / train_b['LotArea'].clip(lower=1)

# 総ポーチ面積
train_b['TotalPorch'] = (train_b['OpenPorchSF'] + train_b['EnclosedPorch']
                         + train_b['3SsnPorch']  + train_b['ScreenPorch'])

area_feats = ['TotalSF', 'LivLotRatio', 'TotalPorch', 'GrLivArea', '1stFlrSF']
corr_area = train_b[area_feats + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('面積系特徴量と SalePrice の相関係数:')
for c, v in corr_area.sort_values(ascending=False).items():
    print(f'  {c:<20}: r = {v:.3f}')


In [ ]:
# ── B-2：バスルーム合計 ───────────────────────────────────────────────────
train_b2 = train.copy()

# 重み付き合計（フルバス=1, ハーフバス=0.5）
train_b2['TotalBath'] = (train_b2['FullBath']
                         + 0.5 * train_b2['HalfBath']
                         + train_b2['BsmtFullBath']
                         + 0.5 * train_b2['BsmtHalfBath'])

r_bath = train_b2['TotalBath'].corr(train_b2['SalePrice'])
print(f'TotalBath と SalePrice の相関係数: r = {r_bath:.3f}')
print()

# バスルーム数ごとの中央価格
print('TotalBath ごとの中央 SalePrice:')
print(train_b2.groupby('TotalBath')['SalePrice'].median().round(0).to_string())


In [ ]:
# ── B-3：リフォーム年数（HouseAge は学生版で実装済みなので RemodAge を追加）───
train_b3 = train.copy()
train_b3['HouseAge']    = train_b3['YrSold'] - train_b3['YearBuilt']     # 学生版で実装済み
train_b3['RemodAge']    = train_b3['YrSold'] - train_b3['YearRemodAdd']  # ← 追加候補
train_b3['IsRemodeled'] = (train_b3['YearRemodAdd'] > train_b3['YearBuilt']).astype(int)

age_feats = ['HouseAge', 'RemodAge', 'IsRemodeled']
corr_age = train_b3[age_feats + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('築年数系特徴量と SalePrice の相関係数:')
for c, v in corr_age.items():
    print(f'  {c:<20}: r = {v:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(train_b3['HouseAge'],  np.log1p(train_b3['SalePrice']), alpha=0.3, s=10)
axes[0].set_xlabel('HouseAge'); axes[0].set_ylabel('log(SalePrice)')
axes[0].set_title(f'HouseAge  (r={corr_age["HouseAge"]:.3f})')
axes[1].scatter(train_b3['RemodAge'],  np.log1p(train_b3['SalePrice']), alpha=0.3, s=10, color='orange')
axes[1].set_xlabel('RemodAge'); axes[1].set_ylabel('log(SalePrice)')
axes[1].set_title(f'RemodAge  (r={corr_age["RemodAge"]:.3f})')
plt.tight_layout(); plt.show()


In [ ]:
# ── B-4：品質 × 面積（強力な組み合わせ） ────────────────────────────────
train_b4 = train.copy()

train_b4['QualxArea']     = train_b4['OverallQual'] * train_b4['GrLivArea']
train_b4['QualxTotalSF']  = train_b4['OverallQual'] * (
    train_b4['1stFlrSF'] + train_b4['2ndFlrSF'] + train_b4['TotalBsmtSF'])
train_b4['CondxArea']     = train_b4['OverallCond'] * train_b4['GrLivArea']

qual_feats = ['QualxArea', 'QualxTotalSF', 'CondxArea', 'OverallQual', 'GrLivArea']
corr_qual = train_b4[qual_feats + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('品質×面積系特徴量と SalePrice の相関係数:')
for c, v in corr_qual.sort_values(ascending=False).items():
    print(f'  {c:<20}: r = {v:.3f}')
print()
print('【TAメモ】QualxArea は r≈0.87 と非常に強く、単体の特徴量を凌駕する。')
print('          一方で OverallQual と多重共線性が高くなる点も指摘できるとよい。')


In [ ]:
# ── B-5：バイナリ・フラグ系（ドメイン知識） ─────────────────────────────
train_b5 = train.copy()

train_b5['HasGarage']     = (train_b5['GarageCars']  > 0).astype(int)
train_b5['HasBsmt']       = (train_b5['TotalBsmtSF'] > 0).astype(int)
train_b5['HasPool']       = (train_b5['PoolArea']     > 0).astype(int)
train_b5['Has2ndFloor']   = (train_b5['2ndFlrSF']     > 0).astype(int)
train_b5['HasFireplace']  = (train_b5['Fireplaces']   > 0).astype(int)
train_b5['LuxurySignal']  = ((train_b5['PoolArea'] > 0) | (train_b5['Fireplaces'] > 0)).astype(int)

flag_feats = ['HasGarage', 'HasBsmt', 'HasPool', 'Has2ndFloor', 'HasFireplace', 'LuxurySignal']
corr_flag = train_b5[flag_feats + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('フラグ系特徴量と SalePrice の相関係数:')
for c, v in corr_flag.sort_values(ascending=False).items():
    print(f'  {c:<20}: r = {v:.3f}')
print()
print('【TAメモ】HasPool の相関が低いのは、プールのある物件が稀なため。')
print('          一方 HasGarage・HasBsmt は住宅価値に直結するため効きやすい。')


In [ ]:
# ── B-6：学生版の add_features() に追加する場合の完全版（TA参照用） ──────
# 学生版 v2 の add_features() は最初から HouseAge を作成済みなので、
# その下に以下のような行を追加すればよい、という形で見せる例

def add_features_full(df):
    """受講生への模範解答・解説用。すべての実装例を一括適用。"""
    df = df.copy()

    # ── ベースの特徴量（学生版にすでにある）────────────────────────
    df['HouseAge']      = df['YrSold'] - df['YearBuilt']

    # ── ↓ ここから追加する例 ──────────────────────────────────────
    # 面積系
    df['TotalSF']       = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['TotalPorch']    = (df['OpenPorchSF'] + df['EnclosedPorch']
                           + df['3SsnPorch']  + df['ScreenPorch'])
    df['LivLotRatio']   = df['GrLivArea'] / df['LotArea'].clip(lower=1)

    # バスルーム
    df['TotalBath']     = (df['FullBath'] + 0.5*df['HalfBath']
                           + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])

    # 築年数（リフォーム関連を追加）
    df['RemodAge']      = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodeled']   = (df['YearRemodAdd'] > df['YearBuilt']).astype(int)

    # 品質×面積
    df['QualxArea']     = df['OverallQual'] * df['GrLivArea']
    df['QualxTotalSF']  = df['OverallQual'] * df['TotalSF']

    # フラグ
    df['HasGarage']     = (df['GarageCars']  > 0).astype(int)
    df['HasBsmt']       = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace']  = (df['Fireplaces']  > 0).astype(int)
    df['LuxurySignal']  = ((df['PoolArea'] > 0) | (df['Fireplaces'] > 0)).astype(int)

    return df

train_full = add_features_full(train)
new_cols = [c for c in train_full.columns if c not in train.columns]
print(f'追加列数: {len(new_cols)}')
corr_all = train_full[new_cols + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('\n追加特徴量と SalePrice の相関係数（降順）:')
for c, v in corr_all.sort_values(ascending=False).items():
    bar = '█' * int(abs(v) * 20)
    print(f'  {c:<20}: r = {v:+.3f}  {bar}')


---
## C. 特徴量選択の実装例

受講生が操作できる変数は `THRESHOLD` と `DROP_COLS` の 2 つです。
以下に閾値の効果確認・多重共線性の検出・削除方針を示します。

### TA向け解説

**閾値（相関フィルタ）の考え方：**
- THRESHOLD が高すぎる → 重要な特徴量まで削除、過学習防止効果は高い
- THRESHOLD が低すぎる → ノイズ特徴量が残る、線形モデルでは過学習しやすい
- 目安: `0.05〜0.15` の範囲が多い。RidgeやLassoなら不要なこともある

**多重共線性の目安：**
- |r| > 0.9：片方は確実に削除候補
- |r| > 0.8：削除を検討
- |r| 0.7〜0.8：ドメイン知識で判断

> **TAメモ**：学生版 v2 の `THRESHOLD` 初期値は `0`（フィルタなし）です。
> 「まず特徴量を追加 → そのあと閾値を上げて削除」という順番を伝えるとよいでしょう。


In [ ]:
# ── C-1：閾値を変えたときの特徴量数の変化 ───────────────────────────────
y_log = np.log1p(train['SalePrice'])
X_num = train.select_dtypes(include='number').drop(columns=['SalePrice', 'Id'])
corr_target = X_num.corrwith(y_log).abs()

print('閾値 → 残る特徴量数:')
for thresh in [0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25]:
    n_keep = (corr_target >= thresh).sum()
    n_drop = (corr_target <  thresh).sum()
    print(f'  THRESHOLD={thresh:.2f}  残る: {n_keep:3d}列  削除: {n_drop:2d}列')

print()
# 削除候補の特徴量を表示（THRESHOLD=0.1の場合）
thresh_show = 0.1
weak = corr_target[corr_target < thresh_show].sort_values()
print(f'THRESHOLD={thresh_show} で削除される特徴量({len(weak)}個):')
for col, v in weak.items():
    print(f'  {col:<25}: |r| = {v:.3f}')


In [ ]:
# ── C-2：多重共線性の確認（高相関ペアの抽出）────────────────────────────
X_num2 = train.select_dtypes(include='number').drop(columns=['SalePrice', 'Id'])
corr_all2 = X_num2.corr()

high_corr_pairs = []
cols = corr_all2.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r_val = corr_all2.iloc[i, j]
        if abs(r_val) >= 0.8:
            high_corr_pairs.append((cols[i], cols[j], round(r_val, 3)))

high_corr_pairs.sort(key=lambda x: -abs(x[2]))
print(f'|r| >= 0.8 の多重共線性ペア ({len(high_corr_pairs)} 組):')
print()
for a, b, r in high_corr_pairs:
    print(f'  {a:<22} × {b:<22}  r = {r:+.3f}')

print()
print('【TAメモ】GarageCars×GarageAreaは最も典型的なペア。')
print('「どちらを残しますか？」と受講生に問うと議論が盛り上がります。')
print('（一般に Cars のほうが SalePrice との相関が高いので Cars 残しが多い）')


In [ ]:
# ── C-3：多重共線性ヒートマップ（代表的なグループ）────────────────────────
mc_groups = {
    'ガレージ系': ['GarageCars', 'GarageArea'],
    '地下室系':   ['TotalBsmtSF', 'BsmtFinSF1', '1stFlrSF'],
    '居住面積系': ['GrLivArea', 'TotRmsAbvGrd', '1stFlrSF'],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (group_name, group_cols) in zip(axes, mc_groups.items()):
    corr_g = train[group_cols].corr()
    sns.heatmap(corr_g, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
                vmin=-1, vmax=1, linewidths=0.5)
    ax.set_title(group_name)
plt.suptitle('代表的な多重共線性グループ', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── C-4：DROP_COLS の推奨例と理由 ─────────────────────────────────────────
print('DROP_COLS の推奨候補と理由:')
drop_candidates = [
    ('GarageArea',     'GarageCars と r=0.88。Cars のほうが SalePrice 相関が高い'),
    ('TotRmsAbvGrd',   'GrLivArea と r=0.83。面積と部屋数は重複情報が多い'),
    ('1stFlrSF',       'TotalBsmtSF と r=0.82。TotalSF を作れば不要になる'),
    ('GarageYrBlt',    'YearBuilt と r=0.83。築年数は HouseAge に統合できる'),
    ('MiscVal',        'SalePrice との相関が極めて低い（|r|<0.02）'),
    ('PoolArea',       '欠損なしだがプール保有率が低く情報量が少ない'),
]
for col, reason in drop_candidates:
    print(f'  "{col}"')
    print(f'    → {reason}')
    print()


In [ ]:
# ── C-5：相関フィルタ + 多重共線性除去を組み合わせた完全版 ────────────────
def select_features(X_df, y_series, threshold=0.1, drop_cols=None):
    """
    Parameters
    ----------
    X_df      : 特徴量 DataFrame（get_dummies 適用済み）
    y_series  : 目的変数 Series
    threshold : 相関フィルタの閾値
    drop_cols : 多重共線性等で直接削除する列リスト
    """
    if drop_cols is None:
        drop_cols = []

    corr = X_df.corrwith(y_series).abs()
    keep = corr[corr >= threshold].index.tolist()
    keep = [c for c in keep if c not in drop_cols]
    return X_df[keep]

# 動作確認
print('select_features 関数のデモ:')
_dummy_X = pd.get_dummies(
    train.drop(columns=['SalePrice','Id']).select_dtypes(include='number'),
    drop_first=True
)
_y = np.log1p(train['SalePrice'])
_selected = select_features(_dummy_X, _y, threshold=0.1,
                             drop_cols=['GarageArea', 'TotRmsAbvGrd'])
print(f'  元の特徴量数: {_dummy_X.shape[1]}')
print(f'  選択後     : {_selected.shape[1]}')


---
## D. パイプライン実行（TA用完全版）

以下は受講生ノートブックの自動評価セル（A-1 / A-2 / B / C を統合したパイプライン）に
対応する TA 参照版です。`add_features_full`、`GROUP_FILL`、推奨 `DROP_COLS` を
組み合わせた場合の参考スコアを示します。

構造は受講生ノートブックのパイプライン実行セルと完全に同じ流れです。
1. (A+B) `add_features_full` で特徴量作成
2. (A-2) `GROUP_FILL` でカテゴリ別代表値補完（get_dummies 前・学習データのみで統計量算出）
3. (A-1) `COL_FILL_MAP` → デフォルト補完 → 「設備なし」NA 処理
4. (C) 相関フィルタ + `DROP_COLS` で特徴量選択
5. 学習・評価


In [ ]:
# TA参照用：推奨設定一式
NUM_FILL_DEFAULT = 'median'
FILL_CONSTANT    = 0
COL_FILL_MAP     = {
    'MasVnrArea': 'constant_0',
}
GROUP_FILL = [
    ('LotFrontage', 'Neighborhood', 'median'),
    ('GarageYrBlt', 'GarageType',   'median'),
]
THRESHOLD  = 0.1
DROP_COLS  = ['GarageArea', 'TotRmsAbvGrd']

print('✅ TA参照用設定')
print(f'   NUM_FILL_DEFAULT = {NUM_FILL_DEFAULT}')
print(f'   COL_FILL_MAP     = {COL_FILL_MAP}')
print(f'   GROUP_FILL       = {GROUP_FILL}')
print(f'   THRESHOLD        = {THRESHOLD}')
print(f'   DROP_COLS        = {DROP_COLS}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TA参照用パイプライン実行（受講生ノートブックのパイプライン実行セルと同じ構造）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── (A+B) 特徴量作成 ──────────────────────────────────────────────────
train_d = add_features_full(train)
test_d  = add_features_full(test)

y_d      = np.log1p(train_d['SalePrice'])
X_d      = train_d.drop(columns=['SalePrice', 'Id'])
X_test_d = test_d.drop(columns=['Id']).copy()

all_d = pd.concat([X_d, X_test_d], axis=0).reset_index(drop=True)

# ── (A-2) カテゴリ別代表値補完 ────────────────────────────────────────
# ⚠ get_dummies 前に行う（グループ列がまだ文字列で存在している）
# グループ統計量は学習データ部分のみから計算（リーク防止）
_train_part_d = all_d.iloc[:len(X_d)]
for _tc, _gc, _af in GROUP_FILL:
    if _tc not in all_d.columns or _gc not in all_d.columns:
        print(f'   ⚠ GROUP_FILL スキップ: "{_tc}" か "{_gc}" が見つかりません')
        continue
    _miss_before = all_d[_tc].isnull().sum()
    if _miss_before == 0:
        continue
    _group_stats = _train_part_d.groupby(_gc)[_tc].agg(_af)
    _fill_vals   = all_d[_gc].map(_group_stats)
    all_d[_tc]   = all_d[_tc].fillna(_fill_vals)
    _miss_after  = all_d[_tc].isnull().sum()
    print(f'   ✅ GROUP_FILL: {_tc} ← {_gc} 別 {_af}  ({_miss_before} → {_miss_after} 件)')

# ── (A-1) カラム別オーバーライド補完 ─────────────────────────────────
for _col, _method in COL_FILL_MAP.items():
    if _col not in all_d.columns or all_d[_col].isnull().sum() == 0:
        continue
    if _method == 'median':
        all_d[_col] = all_d[_col].fillna(all_d[_col].median())
    elif _method == 'mean':
        all_d[_col] = all_d[_col].fillna(all_d[_col].mean())
    elif _method == 'most_frequent':
        all_d[_col] = all_d[_col].fillna(all_d[_col].mode()[0])
    elif _method.startswith('constant_'):
        _cval = float(_method.split('_', 1)[1])
        all_d[_col] = all_d[_col].fillna(_cval)
    else:
        print(f'   ⚠ COL_FILL_MAP: "{_col}" の補完方法 "{_method}" が不明。スキップ')
        continue
    print(f'   ✅ COL_FILL_MAP: {_col} → {_method}（残余欠損: {all_d[_col].isnull().sum()} 件）')

# ── (A-1) デフォルト補完（未補完の数値列に適用）────────────────────────
_num_cols_d = all_d.select_dtypes(include='number').columns
for _col in _num_cols_d:
    if all_d[_col].isnull().sum() == 0:
        continue
    if NUM_FILL_DEFAULT == 'median':
        all_d[_col] = all_d[_col].fillna(all_d[_col].median())
    elif NUM_FILL_DEFAULT == 'mean':
        all_d[_col] = all_d[_col].fillna(all_d[_col].mean())
    elif NUM_FILL_DEFAULT == 'constant':
        all_d[_col] = all_d[_col].fillna(FILL_CONSTANT)

# ── 「設備なし」NA の補完 ──────────────────────────────────────────────
_na_none_d = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
              'GarageType','GarageFinish','GarageQual','GarageCond',
              'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
for _col in _na_none_d:
    if _col in all_d.columns:
        all_d[_col] = all_d[_col].fillna('None')
for _col in all_d.select_dtypes(include='object').columns:
    all_d[_col] = all_d[_col].fillna('Missing')

all_d = pd.get_dummies(all_d, drop_first=True)

# ── (C) 特徴量選択 ────────────────────────────────────────────────────
Xd_train_proc = all_d.iloc[:len(X_d)].copy()
yd_corr = Xd_train_proc.corrwith(y_d).abs()
keep_d  = yd_corr[yd_corr >= THRESHOLD].index.tolist()
keep_d  = [_col for _col in keep_d if _col not in DROP_COLS]

Xd_proc      = all_d[keep_d].iloc[:len(X_d)].copy()
Xd_test_proc = all_d[keep_d].iloc[len(X_d):].copy()

print(f'   特徴量数: {all_d.shape[1]} → {len(keep_d)} 列')

# ── (D) 学習・評価 ────────────────────────────────────────────────────
Xd_tr, Xd_vl, yd_tr, yd_vl = train_test_split(Xd_proc, y_d, test_size=0.2, random_state=42)
model_ta = LinearRegression()
model_ta.fit(Xd_tr, yd_tr)
rmse_ta = np.sqrt(mean_squared_error(yd_vl, model_ta.predict(Xd_vl)))

print()
print(f'🔵 ベースライン RMSE  : {rmse_baseline:.5f}')
print(f'🌟 TA 参照版  RMSE   : {rmse_ta:.5f}')
print(f'   改善率: {(rmse_baseline - rmse_ta)/rmse_baseline*100:+.2f}%')


In [ ]:
# ── ボーナス：Ridge / Lasso への切り替え（上級者向けヒント）────────────────
results = {}
for name, model in [('LinearRegression', LinearRegression()),
                    ('Ridge(alpha=10)',   Ridge(alpha=10)),
                    ('Ridge(alpha=100)',  Ridge(alpha=100)),
                    ('Lasso(alpha=0.001)',Lasso(alpha=0.001, max_iter=10000))]:
    m = model
    m.fit(Xd_tr, yd_tr)
    rmse = np.sqrt(mean_squared_error(yd_vl, m.predict(Xd_vl)))
    results[name] = rmse

print('モデル比較（同じ特徴量）:')
for name, rmse in sorted(results.items(), key=lambda x: x[1]):
    bar = '█' * int((0.2 - rmse) * 500)
    print(f'  {name:<30}: RMSE={rmse:.5f}  {bar}')
print()
print('【TAメモ】Ridge は多重共線性があるデータで安定する傾向があります。')
print('          時間に余裕があれば「正則化」の概念を簡単に紹介してみてください。')


---
## E. 提出ファイル作成


In [ ]:
test_pred_log = model_ta.predict(Xd_test_proc)
test_pred     = np.expm1(test_pred_log)

submission_ta = pd.DataFrame({'Id': test_ids, 'SalePrice': test_pred})
submission_ta.to_csv('submission_ta.csv', index=False)
print('✅ submission_ta.csv を作成しました')
print(submission_ta.describe().round(0))


---
## 📝 TA 向け指導メモまとめ

### 受講生がつまずきやすいポイント（v2 で追加された A-2 を含む）

| つまずき | 対処法 |
|:---|:---|
| `fillna` の引数に何を入れればいいかわからない | `df[col].median()` を渡すことを示す |
| `COL_FILL_MAP` の `constant_<数値>` の書式がわからない | `'constant_0'` のように **アンダースコアの後に数値** と説明する |
| `GROUP_FILL` のタプルの順序を間違える | `(補完したい列, グループ化する列, 集計関数)` の順だと毎回繰り返す |
| `GROUP_FILL` を学習データだけで計算する理由が分からない | 「テストデータの統計量を使うとリーク（不正なヒント）になる」と説明する |
| `add_features` に何を書けばいいかわからない | B-4 の `QualxArea` を一例として見せる |
| 閾値を変えても RMSE が改善しない | 特徴量を先に追加してから閾値を調整する順番を伝える |
| `DROP_COLS` を空リストにしている | 多重共線性のヒートマップを見せて `GarageArea` を消すよう促す |

### スコア目安（LinearRegression）

| 設定 | 期待 RMSE |
|:---|:---|
| ベースライン（前回） | 約 0.155〜0.165 |
| 特徴量追加のみ | 約 0.140〜0.155 |
| 特徴量追加 + GROUP_FILL + 選択最適化 | 約 0.130〜0.145 |
| Ridge に切り替え | 約 0.125〜0.140 |

> 数値は環境・random_state によって若干変わります。
